# 04 Generation

`src.generation` RAG 파이프라인을 구동하고 `src.evaluation` 답변 생성 품질을 hit / recall / bert score 기준 평가.



## Colab Setup

In [1]:
!nvidia-smi

Tue May  5 13:45:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/songhahyun/finance-terms-rag-chatbot.git'
REPO_BRANCH = 'dev'
REPO_DIR = Path('/content/finance-terms-rag-chatbot')

In [3]:
!git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
print('cwd =', Path.cwd())

fatal: destination path '/content/finance-terms-rag-chatbot' already exists and is not an empty directory.
/content/finance-terms-rag-chatbot
cwd = /content/finance-terms-rag-chatbot


In [ ]:
# Python deps
!pip install -q -U pip
!pip install -q -r requirements.txt

# Ollama install + serve
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pandas tqdm

# Ollama pull
!ollama pull deepseek-r1:1.5b
!ollama pull llama3.2:3b

os.environ['OLLAMA_BASE_URL'] = 'http://127.0.0.1:11434'
os.environ['OLLAMA_SMALL_MODEL'] = 'deepseek-r1:1.5b'
os.environ['OLLAMA_COMPLEX_MODEL'] = 'llama3.2:3b'

print('Ollama ready:', os.environ['OLLAMA_BASE_URL'])

## Run answer generation

In [ ]:
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
import pandas as pd

from src.common.config import get_settings
from src.evaluation.pipeline import run_generation_experiment

settings = get_settings()

# EVAL_CSV_PATH = ROOT / "data" / "eval" / "testset" / "golden_testset_v2.csv"
EVAL_CSV_PATH = ROOT / "data" / "eval" / "testset" / "test.csv"
CHUNK_JSON_PATH = settings.default_chunk_json_path
BM25_INDEX_PATH = CHUNK_JSON_PATH.with_name("bm25_index.pkl")
OUTPUT_DIR = ROOT / "data" / "eval" / "outputs" / "generation_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = [
    {
        "name": "hybrid_clova_bge-m3",
        "retrieval_mode": "hybrid",
        "dense_provider": "clova",
        "dense_model_name": "bge-m3",
        "dense_collection_name": "docs_clova",
        "dense_persist_directory": str(settings.chroma_clova_dir),
    },
    {
        "name": "dense_clova_bge-m3",
        "retrieval_mode": "dense",
        "dense_provider": "clova",
        "dense_model_name": "bge-m3",
        "dense_collection_name": "docs_clova",
        "dense_persist_directory": str(settings.chroma_clova_dir),
    },
    {
        "name": "hybrid_openai_text-embedding-3-small",
        "retrieval_mode": "hybrid",
        "dense_provider": "openai",
        "dense_model_name": "text-embedding-3-small",
        "dense_collection_name": "docs_openai",
        "dense_persist_directory": str(settings.chroma_openai_dir),
    },
]

print("Eval CSV:", EVAL_CSV_PATH)
print("Chunk JSON:", CHUNK_JSON_PATH)
print("BM25 index PKL:", BM25_INDEX_PATH)
print("Output Dir:", OUTPUT_DIR)
print("Stage models:")
print("- Stage 1 / 2-b / Stage3(simple):", settings.ollama_small_model)
print("- Stage3(complex):", settings.ollama_complex_model)


In [ ]:
all_results = {}
summary_rows = []

for exp in EXPERIMENTS:
    output_path = OUTPUT_DIR / f"{exp['name']}.csv"
    result_df = run_generation_experiment(
        experiment_name=exp["name"],
        eval_csv_path=EVAL_CSV_PATH,
        chunk_json_path=CHUNK_JSON_PATH,
        output_csv_path=output_path,
        retrieval_mode=exp["retrieval_mode"],
        dense_provider=exp["dense_provider"],
        dense_model_name=exp["dense_model_name"],
        dense_collection_name=exp["dense_collection_name"],
        dense_persist_directory=exp["dense_persist_directory"],
        bm25_index_path=BM25_INDEX_PATH,
        ollama_small_model=settings.ollama_small_model,
        ollama_complex_model=settings.ollama_complex_model,
        ollama_base_url=settings.ollama_base_url,
        ollama_timeout=settings.ollama_timeout,
        k=5,
        language="ko",
    )
    all_results[exp["name"]] = output_path

    summary_rows.append(
        {
            "experiment": exp["name"],
            "rows": len(result_df),
            "avg_recall": float(result_df["recall"].mean()) if len(result_df) else 0.0,
            "avg_mrr": float(result_df["mrr"].mean()) if len(result_df) else 0.0,
            "avg_stage_1_latency_sec": float(result_df["stage_1_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage_2a_latency_sec": float(result_df["stage_2a_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage_2b_latency_sec": float(result_df["stage_2b_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage_3_latency_sec": float(result_df["stage_3_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_total_latency_sec": float(result_df["total_latency_sec"].mean()) if len(result_df) else 0.0,
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_output_path = OUTPUT_DIR / "generation_experiment_summary.csv"
summary_df.to_csv(summary_output_path, index=False, encoding="utf-8-sig")

print("Saved output files:")
for name, path in all_results.items():
    print(f"- {name}: {path}")
print(f"- summary: {summary_output_path}")

summary_df
